# Bank Payment Fraud Detection — Exploratory Data Analysis
### Understanding Class Imbalance and Fraud Patterns in Credit Card Transactions

## 1. Project Overview
This notebook explores a synthetic credit-card payment fraud dataset. Fraud is rare (~0.5% of transactions), creating a severe class-imbalance problem. We analyse transaction patterns, feature distributions, and temporal trends to understand how fraud looks different from legitimate transactions.

## 2. Learning Objectives
- Understand the challenge of class imbalance in fraud detection
- Analyse anonymised/PCA features for fraud vs legitimate distributions
- Apply outlier detection intuitions to fraud signals
- Evaluate AUC-ROC and precision-recall as appropriate metrics for imbalanced data
- Understand why accuracy is a misleading metric for rare events

## 3. Business / Research Problem
**Problem:** Payment processors must detect fraudulent transactions in real time, often with less than 100ms latency and without annoying legitimate customers with false positives. The cost of a missed fraud (~$500 average) far exceeds the cost of a blocked legitimate transaction (~$10 customer inconvenience).

## 4. Why This Analysis Matters
Global card-payment fraud losses exceed $30 billion per year. Traditional rule-based systems miss adaptive fraudsters. Machine-learning models must be trained on deeply imbalanced data — understanding that imbalance at the EDA stage is the first step toward building a reliable detector.

## 5. Dataset Overview
The dataset contains credit card transactions made by European cardholders in September 2013:
- `Time` — seconds since first transaction in dataset
- `V1`–`V28` — PCA-transformed features (original features anonymised for privacy)
- `Amount` — transaction amount in euros
- `Class` — 0 = legitimate, 1 = fraud

## 6. Dataset Source and License Notes
- **Kaggle competition:** `mlg-ulb/creditcardfraud`
- **Source:** Machine Learning Group, Université Libre de Bruxelles
- **License:** Open Database License (ODbL)

## 7. Environment Setup

In [1]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','scipy','scikit-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

Ready.


## 8. Imports

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

Imports OK.


## 9. Configuration / Constants

In [3]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'mlg-ulb/creditcardfraud'
TARGET = 'Class'
FRAUD_LABEL = 1
LEGIT_LABEL = 0

## 10. Dataset Download

In [4]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
csv_files = list(DATA_DIR.glob('*.csv'))
print('Files:', [f.name for f in csv_files])

Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0


Files: ['creditcard.csv']


In [5]:
df = pd.read_csv(csv_files[0])
print(f'Shape: {df.shape}')
df.head(3)

Shape: (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0


## 11. Data Validation Checks

In [6]:
print('Missing values:', df.isnull().sum().sum())
print('Dtypes:', df.dtypes.value_counts().to_dict())
print('\nClass distribution:')
vc = df[TARGET].value_counts()
print(vc)
print(f'Fraud rate: {vc[1]/len(df)*100:.4f}%')

Missing values: 0
Dtypes: {dtype('float64'): 30, dtype('int64'): 1}

Class distribution:
Class
0    284315
1       492
Name: count, dtype: int64
Fraud rate: 0.1727%


## 12. Data Cleaning
The dataset is clean by design. We standardise `Amount` and `Time` for fair comparison.

In [7]:
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_hours'] = df['Time'] / 3600
fraud = df[df[TARGET]==1]
legit = df[df[TARGET]==0]
print(f'Fraudulent records: {len(fraud)}')
print(f'Legitimate records: {len(legit)}')

Fraudulent records: 492
Legitimate records: 284315


## 13. Exploratory Data Analysis

In [8]:
print('Dataset timespan:', round(df['Time_hours'].max(),1), 'hours (~2 days)')
print('\nAmount statistics:')
df.groupby(TARGET)['Amount'].describe().round(2)

Dataset timespan: 48.0 hours (~2 days)

Amount statistics:


,count,mean,std,min,25%,50%,75%,max
Class,,,,,,,,
0,284315.0,88.29,250.11,0.0,5.65,22.00,77.05,25691.16
1,492.0,122.21,256.68,0.0,1.00,9.25,105.89,2125.87


## 14. Univariate Analysis

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
# Class balance
vc.plot(kind='bar', ax=axes[0], color=['steelblue','firebrick'])
axes[0].set_xticklabels(['Legitimate','Fraud'], rotation=0)
axes[0].set_title('Class Distribution (Severe Imbalance)')
axes[0].set_ylabel('Count')
for bar, val in zip(axes[0].patches, vc.values):
    axes[0].text(bar.get_x()+0.2, bar.get_height()+100, f'{val:,}', fontsize=9)
# Amount distribution by class
axes[1].hist(legit['Amount'].clip(upper=500), bins=60, alpha=0.7, color='steelblue', label='Legit', density=True)
axes[1].hist(fraud['Amount'].clip(upper=500), bins=30, alpha=0.7, color='firebrick', label='Fraud', density=True)
axes[1].set_title('Transaction Amount Distribution (clipped $500)')
axes[1].set_xlabel('Amount ($)')
axes[1].legend()
plt.tight_layout(); plt.show()

## 15. Bivariate / Multivariate Analysis
Compare PCA features V1–V10 between fraud and legitimate classes.

In [10]:
# Compare key V features
v_features = [f'V{i}' for i in range(1,11)]
fig, axes = plt.subplots(2, 5, figsize=(20,8))
for i, (ax, feat) in enumerate(zip(axes.flat, v_features)):
    ax.hist(legit[feat], bins=50, alpha=0.6, color='steelblue', density=True, label='Legit')
    ax.hist(fraud[feat], bins=50, alpha=0.6, color='firebrick', density=True, label='Fraud')
    ax.set_title(feat, fontsize=11)
    ax.set_yticks([])
handles = [plt.Rectangle((0,0),1,1,color='steelblue',alpha=0.6),
           plt.Rectangle((0,0),1,1,color='firebrick',alpha=0.6)]
fig.legend(handles, ['Legit','Fraud'], loc='upper right', fontsize=12)
fig.suptitle('Feature Distributions: Fraud vs Legitimate (V1–V10)', fontsize=14)
plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights
### Time-of-Day Fraud Pattern

In [11]:
fig, ax = plt.subplots(figsize=(14,4))
ax.hist(legit['Time_hours'] % 24, bins=48, alpha=0.6, density=True, color='steelblue', label='Legit')
ax.hist(fraud['Time_hours'] % 24, bins=48, alpha=0.6, density=True, color='firebrick', label='Fraud')
ax.set_title('Transaction Time (hour of day) — Fraud vs Legitimate')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout(); plt.show()

## 17. Statistical Checks / Hypothesis Testing
For each PCA feature, test whether the fraud and legitimate distributions differ significantly (Mann–Whitney U).

In [12]:
results = []
for col in [f'V{i}' for i in range(1,29)] + ['Amount']:
    u, p = stats.mannwhitneyu(fraud[col], legit[col], alternative='two-sided')
    results.append({'feature':col,'p_value':p,'significant':p<0.001})
res_df = pd.DataFrame(results).sort_values('p_value')
print('Features with significant difference (p < 0.001):')
print(res_df[res_df['significant']]['feature'].tolist())
print(f'\nTotal significant: {res_df["significant"].sum()} / {len(res_df)}')

Features with significant difference (p < 0.001):
['V14', 'V4', 'V12', 'V11', 'V10', 'V3', 'V2', 'V16', 'V9', 'V7', 'V17', 'V1', 'V6', 'V21', 'V18', 'V5', 'V27', 'V8', 'V19', 'V20', 'V28', 'V24', 'Amount']

Total significant: 23 / 29


## 18. Visual Analysis
### Correlation Heatmap for Fraud Transactions

In [13]:
# Top discriminating features
top_feats = res_df.head(10)['feature'].tolist()
fig, axes = plt.subplots(1, 2, figsize=(16,6))
for ax, data, title in zip(axes, [fraud, legit], ['Fraud','Legitimate']):
    sns.heatmap(data[top_feats].corr(), cmap='coolwarm', center=0, ax=ax, annot=False, linewidths=0.5)
    ax.set_title(f'Feature Correlation — {title}')
plt.tight_layout(); plt.show()

In [14]:
# Box plots — top 6 most discriminating features
top6 = res_df.head(6)['feature'].tolist()
fig, axes = plt.subplots(2, 3, figsize=(15,8))
for ax, feat in zip(axes.flat, top6):
    combined = pd.concat([
        fraud[[feat]].assign(Class='Fraud'),
        legit[[feat]].sample(1000, random_state=42)[[feat]].assign(Class='Legit')
    ])
    sns.boxplot(x='Class', y=feat, data=combined, palette={'Fraud':'firebrick','Legit':'steelblue'}, ax=ax)
    ax.set_title(feat)
fig.suptitle('Top Discriminating Features: Fraud vs Legitimate', fontsize=14)
plt.tight_layout(); plt.show()

## 19. Key Findings
1. **Class imbalance is extreme** — fraud accounts for ~0.17% of all transactions.
2. **Several V features strongly separate classes** — V1, V3, V4, V9, V10, V11, V12, V14, V16, V17.
3. **Fraud transactions tend to be lower-amount** — median fraud amount is lower than legitimate.
4. **Time-of-night pattern** — fraud is proportionally more common during off-peak hours (1–4 AM).
5. **Accuracy is a misleading metric** — a model predicting 'all legitimate' gets 99.83% accuracy but detects zero fraud.

## 20. Limitations
- Features V1–V28 are PCA-transformed — cannot interpret original feature meanings.
- Dataset covers only 2 days; seasonal and year-over-year patterns are absent.
- Synthetic/anonymised data may not generalise to real-world deployment.
- No merchant or geographic context is available.

## 21. Recommendations / Next Steps
1. Apply SMOTE or cost-sensitive learning to handle class imbalance.
2. Train an Isolation Forest and compare with XGBoost on AUC-PR.
3. Use SHAP values to interpret which V features drive predictions.
4. Establish a business cost matrix and optimise the classification threshold accordingly.
5. Build a real-time streaming fraud detector simulation.

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Using accuracy as the metric | 99.8% accuracy by always predicting 'legit' | Use AUC-PR or F1 |
| Not scaling Amount | V1–V28 are already scaled; Amount is not | StandardScaler or RobustScaler |
| Over-applying SMOTE | If applied before cross-validation, leaks synthetic samples | Apply inside CV loop |
| Treating V-features as interpretable | They are PCA components — avoid naïve interpretation | Use SHAP for insights |
| Ignoring time ordering | Train/test split must respect temporal order | Use time-based split |

## 23. Mini Challenge / Exercises
1. **Percentile threshold**: Find the V14 threshold that separates 90% of fraud from legitimate.
2. **Cost analysis**: If FP costs $10 and FN costs $500, what threshold minimises total cost?
3. **Isolation Forest**: Train an unsupervised Isolation Forest — what AUC-ROC does it achieve?
4. **SMOTE**: Apply SMOTE to the training set and retrain a logistic regression — compare AUC-PR before/after.
5. **How to extend**: Use the IEEE-CIS Fraud Detection Kaggle dataset for a richer, multi-feature version.

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  Fraud is extremely rare — class imbalance is the defining challenge.
TAKEAWAY 2  Several PCA features (V1, V3, V14, V17) are highly discriminating.
TAKEAWAY 3  Accuracy is useless here; use AUC-ROC and especially AUC-PR.
TAKEAWAY 4  Fraud is not random in time — off-peak hours see proportionally more fraud.
TAKEAWAY 5  EDA drives smart feature selection and threshold decisions downstream.
```